# FLIDS Day 2 Extension — Kaggle GPU AutoRun

Evaluates four new federated methods under the exact source-locked FLIDS NetFlow protocol (93,810 rows, 4 held-out domains × 5 seeds, 4-candidate inner source-only tuning):
1. **FedLogReg** — Federated Logistic Regression (tests whether centralized linear superiority survives federated training)
2. **FedBN_TTA** — Zero-target-label Unsupervised Test-Time BatchNorm Adaptation (AdaBN on a disjoint 2,000-flow unlabeled stream at $\pi=1\%$)
3. **FedAvg_Tail** — FedAvg with Asymmetric Negative-Tail Focal BCE Loss
4. **FedNova_Tail** — FedNova with Asymmetric Negative-Tail Focal BCE Loss

### Kaggle Setup
1. Turn on **Accelerator: GPU T4 x2** (or P100).
2. Click **Add Input** and attach the kernel output `fragoa/notebook6ceac598dd` (or any dataset containing `FLIDS_Master_V3_Fresh_Results.zip`, `FLIDS_Exp09_Canonical_Input.zip`, or `nf_cache.npz`).
3. Click **Run All**. Output archive `FLIDS_Day2_Methods_Results.zip` and summary CSVs are saved in `/kaggle/working/day2_work/`.

In [ ]:
# Configuration
SMOKE_TEST = False   # Set True for a ~30s sanity check on 1 domain x 1 seed
METHODS = ['FedLogReg', 'FedBN_TTA', 'FedAvg_Tail', 'FedNova_Tail']
WORK_DIR = '/kaggle/working/day2_work'
INPUT_OVERRIDE = None  # Optional explicit path if auto-discovery needs override


In [ ]:
# Write embedded day2_methods.py module
from pathlib import Path
CODE = '"""FLIDS Day 2 Extension: FedLogReg, FedBN_TTA, FedAvg_Tail, and FedNova_Tail.\n\nFollows the exact source-locked protocol of experiments/tuned/master.py:\n- Exact same 93,810-row deduplicated NF-UQ-NIDS-v2 cache and split digests\n- Exact same signed-log + source-fit StandardScaler preprocessing\n- Exact same 4-candidate inner leave-one-source-out selection (max pAUC@5%)\n- Exact same 150-round / min_round=30 / patience=10 / local_steps=8 / batch=256 budget\n- Zero-leakage FedBN_TTA: adapts BatchNorm stats on a disjoint 2,000-flow unlabeled\n  target stream at realistic pi=1% prevalence (zero target labels, zero gradients)\n- Evaluates source-locked 1% FPR, retrospective 1% target-quantile recall,\n  FedRAT, and zero-target-label unsupervised calibration (UT-Quantile, UT-RAT at pi=1%).\n\nUsage:\n    python experiments/day2/day2_methods.py --input /path/to/FLIDS_Master_V3_Fresh_Results.zip --work ./day2_work\n"""\nimport os\nfor _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ[_k] = "1"\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.stats import binom\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom torch import nn\n\nSEEDS = [0, 42, 3407, 123, 2026]\nDOMAINS = ["NF-UNSW-NB15", "NF-ToN-IoT", "NF-BoT-IoT", "NF-CSE-CIC-IDS2018"]\nMETHODS = ["FedLogReg", "FedBN_TTA", "FedAvg_Tail", "FedNova_Tail"]\n\n\ndef make_seed(*args) -> int:\n    return int.from_bytes(hashlib.sha256(repr(args).encode()).digest()[:4], "little")\n\n\ndef make_rng(*args) -> np.random.Generator:\n    return np.random.default_rng(make_seed(*args))\n\n\ndef signed_log(x: np.ndarray) -> np.ndarray:\n    c = np.clip(np.asarray(x, dtype=np.float64), -1e12, 1e12)\n    return np.clip(np.sign(c) * np.log1p(np.abs(c)), -18.0, 18.0).astype(np.float32)\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef save_json(path: Path, obj) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef npwrite(path: Path, **kw) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    with tmp.open("wb") as f:\n        np.savez_compressed(f, **kw)\n    tmp.replace(path)\n\n\ndef emp_cut(scores: np.ndarray, alpha: float) -> float:\n    s = np.sort(np.asarray(scores, dtype=np.float64))\n    if not len(s):\n        return float("inf")\n    k = math.floor(alpha * len(s))\n    return float(np.nextafter(s[len(s) - k - 1], np.inf))\n\n\ndef wilks_cut(scores: np.ndarray, alpha: float, delta: float = 0.05) -> float:\n    s = np.sort(np.asarray(scores, dtype=np.float64))\n    n = len(s)\n    k = int(binom.ppf(1.0 - delta, n, 1.0 - alpha)) + 1\n    return float(np.nextafter(s[k - 1], np.inf)) if k <= n else float("inf")\n\n\ndef fedrat_cut(cal_scores, cal_y, cal_d, alpha=0.01, delta=0.05) -> float:\n    groups = sorted(set(cal_d))\n    cuts = [wilks_cut(cal_scores[(cal_d == g) & (cal_y == 0)], alpha, delta / len(groups)) for g in groups]\n    return max(cuts)\n\n\ndef pauc(y: np.ndarray, s: np.ndarray, max_fpr=None) -> float:\n    return float(roc_auc_score(y, s, max_fpr=max_fpr))\n\n\ndef eval_op(y: np.ndarray, s: np.ndarray, thr: float):\n    n = s[y == 0]\n    p = s[y == 1]\n    fp = int((n >= thr).sum())\n    tp = int((p >= thr).sum())\n    return fp / len(n), tp / len(p), fp, tp\n\n\ndef dedup(x, y, d, f):\n    keys = np.array([hashlib.blake2b(np.ascontiguousarray(v).tobytes(), digest_size=16).hexdigest() for v in x])\n    _, inv, _ = np.unique(keys, return_inverse=True, return_counts=True)\n    bad = set()\n    seen = {}\n    for i, k in enumerate(inv):\n        val = (str(d[i]), int(y[i]), str(f[i]))\n        if k in seen:\n            if seen[k][0] != val:\n                bad.add(k)\n        else:\n            seen[k] = (val, i)\n    return np.array([i for k, (_, i) in seen.items() if k not in bad], dtype=int)\n\n\ndef load_nf_cache(input_path: Path, work: Path):\n    cached = work / "nf_cache.npz"\n    if cached.exists():\n        a = np.load(cached, allow_pickle=False)\n        return a["x"], a["y"], a["d"], a["family"]\n\n    if input_path.is_file() and input_path.suffix == ".npz":\n        a = np.load(input_path, allow_pickle=False)\n        x, y, d, f = a["x"], a["y"], a["d"], a["family"]\n    else:\n        with zipfile.ZipFile(input_path) as z:\n            names = set(z.namelist())\n            if "nf_cache.npz" in names:\n                a = np.load(io.BytesIO(z.read("nf_cache.npz")), allow_pickle=False)\n                x, y, d, f = a["x"], a["y"], a["d"], a["family"]\n            elif "reproducibility/flids_exp02_cache.npz" in names:\n                a = np.load(io.BytesIO(z.read("reproducibility/flids_exp02_cache.npz")), allow_pickle=False)\n                x0, y0, d0, f0 = a["x"], a["y"], a["d"], a["attack"]\n                keep = dedup(x0, y0, d0, f0)\n                x, y, d, f = x0[keep], y0[keep], d0[keep], f0[keep]\n            else:\n                raise FileNotFoundError(\n                    f"Neither nf_cache.npz nor reproducibility/flids_exp02_cache.npz found in {input_path}"\n                )\n    npwrite(cached, x=x, y=y, d=d, family=f)\n    return x, y, d, f\n\n\ndef groupsplit(ids, y, s, frac):\n    a, b = train_test_split(np.asarray(ids), test_size=frac, random_state=s, stratify=y[ids])\n    return np.sort(a), np.sort(b)\n\n\ndef split_outer(y, d, held, s):\n    """Exact same outer split as experiments/tuned/master.py."""\n    src = np.flatnonzero(~np.isin(d, held))\n    test = np.flatnonzero(np.isin(d, held))\n    domains = sorted(set(d[src]))\n    fit, stop, cal, st = [], [], [], []\n    for dom in domains:\n        ids = src[d[src] == dom]\n        a, b = groupsplit(ids, y, make_seed(s, dom, "outer"), 0.30)\n        c, e = groupsplit(b, y, make_seed(s, dom, "cal"), 2 / 3)\n        g, h = groupsplit(e, y, make_seed(s, dom, "st"), 0.5)\n        fit.extend(a)\n        stop.extend(c)\n        cal.extend(g)\n        st.extend(h)\n    parts = tuple(np.array(v, dtype=int) for v in (fit, stop, cal, st, test))\n    assert len(np.unique(np.concatenate(parts))) == sum(map(len, parts))\n    return parts\n\n\ndef candidate_grid(method: str):\n    if method == "FedLogReg":\n        return [\n            dict(lr=0.01, wd=1e-2),\n            dict(lr=0.01, wd=1e-3),\n            dict(lr=0.03, wd=1e-3),\n            dict(lr=0.03, wd=1e-4),\n        ]\n    if method == "FedBN_TTA":\n        return [dict(lr=l) for l in (0.0003, 0.001, 0.003, 0.01)]\n    if method == "FedAvg_Tail":\n        return [\n            dict(lr=0.001, lam=2.0, gamma=2.0),\n            dict(lr=0.001, lam=5.0, gamma=2.0),\n            dict(lr=0.003, lam=2.0, gamma=2.0),\n            dict(lr=0.003, lam=5.0, gamma=2.0),\n        ]\n    if method == "FedNova_Tail":\n        return [\n            dict(lr=0.003, lam=2.0, gamma=2.0),\n            dict(lr=0.003, lam=5.0, gamma=2.0),\n            dict(lr=0.01, lam=2.0, gamma=2.0),\n            dict(lr=0.01, lam=5.0, gamma=2.0),\n        ]\n    raise ValueError(f"Unknown method: {method}")\n\n\nclass LinearLogReg(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Linear(n_in, 1)\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\nclass MLP(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\nclass MLPBN(nn.Module):\n    def __init__(self, n_in: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(n_in, 64),\n            nn.BatchNorm1d(64),\n            nn.ReLU(),\n            nn.Linear(64, 32),\n            nn.BatchNorm1d(32),\n            nn.ReLU(),\n            nn.Linear(32, 1),\n        )\n\n    def forward(self, x):\n        return self.net(x).squeeze(-1)\n\n\ndef tail_weighted_bce(logits: torch.Tensor, targets: torch.Tensor, lam: float, gamma: float = 2.0) -> torch.Tensor:\n    """Asymmetric negative-tail focal BCE: upweights high-scoring benign flows to suppress low-FPR FPs."""\n    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")\n    with torch.no_grad():\n        probs = torch.sigmoid(logits)\n        neg_mask = (targets < 0.5).float()\n        weights = 1.0 + neg_mask * (lam * (probs ** gamma))\n    return (bce * weights).mean()\n\n\ndef score_model(net: nn.Module, xx: torch.Tensor, ids: np.ndarray) -> np.ndarray:\n    net.eval()\n    out = []\n    with torch.no_grad():\n        for chunk in np.array_split(ids, max(1, math.ceil(len(ids) / 4096))):\n            out.append(torch.sigmoid(net(xx[chunk])).cpu().numpy())\n    return np.concatenate(out).astype(np.float64)\n\n\ndef adapt_bn_unsupervised(nets, xx: torch.Tensor, unlabeled_ids: np.ndarray, batch_size: int = 512):\n    """Unsupervised AdaBN: recompute BN running stats from unlabeled target flows only (zero labels, zero grad)."""\n    adapted = []\n    for src_net in nets:\n        net = MLPBN(xx.shape[1]).to(xx.device)\n        net.load_state_dict({k: v.detach().clone() for k, v in src_net.state_dict().items()})\n        for m in net.modules():\n            if isinstance(m, nn.BatchNorm1d):\n                m.reset_running_stats()\n                m.momentum = None\n        net.train()\n        with torch.no_grad():\n            for chunk in np.array_split(unlabeled_ids, max(1, math.ceil(len(unlabeled_ids) / batch_size))):\n                if len(chunk) >= 2:\n                    _ = net(xx[chunk])\n        net.eval()\n        adapted.append(net)\n    return adapted\n\n\ndef fit_method(raw, y, d, tr, va, method, cfg, s, opts):\n    t0 = time.perf_counter()\n    trans = signed_log(raw)\n    sc = StandardScaler().fit(trans[tr])\n    X = np.clip(sc.transform(trans).astype(np.float32), -20.0, 20.0)\n\n    device = opts["device"]\n    xx = torch.as_tensor(X, device=device)\n    yy = torch.as_tensor(y, dtype=torch.float32, device=device)\n    torch.manual_seed(s)\n    r = make_rng(s, method)\n\n    clients = [tr[d[tr] == dom] for dom in sorted(set(d[tr]))]\n    rounds = opts["rounds"]\n    local_steps = opts["local_steps"]\n    check = opts["check_every"]\n    patience = opts["patience"]\n    min_round = opts["min_round"]\n    B = opts["batch"]\n\n    steps = 0\n    comm = 0\n    best = -np.inf\n    best_state = None\n    best_round = 0\n    wait = 0\n\n    if method == "FedBN_TTA":\n        nets = [MLPBN(X.shape[1]).to(device) for _ in clients]\n        init = nets[0].state_dict()\n        for n in nets:\n            n.load_state_dict(init)\n        shared = [k for k in init if k.startswith(("net.0.", "net.3.", "net.6."))]\n        state = {k: init[k].detach().clone() for k in shared}\n\n        def sync():\n            for n in nets:\n                st = n.state_dict()\n                st.update(state)\n                n.load_state_dict(st)\n\n        def pred_src(ids):\n            return np.mean([score_model(n, xx, ids) for n in nets], axis=0)\n\n        for rd in range(1, rounds + 1):\n            sync()\n            w = np.array([len(c) for c in clients], dtype=float)\n            w /= w.sum()\n            updates = []\n            for i, ids in enumerate(clients):\n                n = nets[i]\n                n.train()\n                opt = torch.optim.Adam(n.parameters(), lr=cfg["lr"])\n                for _ in range(local_steps):\n                    b = r.choice(ids, max(2, min(B, len(ids))), replace=len(ids) < 2)\n                    opt.zero_grad()\n                    loss = nn.functional.binary_cross_entropy_with_logits(n(xx[b]), yy[b])\n                    loss.backward()\n                    opt.step()\n                    steps += 1\n                updates.append({k: n.state_dict()[k].detach().clone() for k in shared})\n                comm += 2 * sum(v.numel() * v.element_size() for v in state.values())\n            state = {k: sum(float(a) * u[k] for a, u in zip(w, updates)) for k in shared}\n            sync()\n            if rd % check == 0 or rd == rounds:\n                v = pauc(y[va], pred_src(va), 0.05)\n                if v > best + 1e-5:\n                    best = v\n                    best_round = rd\n                    best_state = [{k: val.detach().cpu().clone() for k, val in n.state_dict().items()} for n in nets]\n                    wait = 0\n                else:\n                    wait += 1\n            if rd >= min_round and wait >= patience:\n                break\n\n        for n, st in zip(nets, best_state):\n            n.load_state_dict(st)\n\n        def pred_tta(ids, adapt_ids=None):\n            if adapt_ids is None:\n                return pred_src(ids)\n            adapted = adapt_bn_unsupervised(nets, xx, adapt_ids)\n            return np.mean([score_model(n, xx, ids) for n in adapted], axis=0)\n\n        info = dict(\n            seconds=time.perf_counter() - t0,\n            steps=steps,\n            bytes=comm,\n            best_round=best_round,\n            rounds_run=rd,\n            source_stop_pauc05=best,\n        )\n        return pred_src, pred_tta, info\n\n    def build_net():\n        return LinearLogReg(X.shape[1]).to(device) if method == "FedLogReg" else MLP(X.shape[1]).to(device)\n\n    model = build_net()\n    mean_client_len = float(np.mean([len(c) for c in clients]))\n\n    for rd in range(1, rounds + 1):\n        state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        weights = np.array([len(c) for c in clients], dtype=float)\n        weights /= weights.sum()\n        deltas = []\n        counts = []\n\n        for ids in clients:\n            local = build_net()\n            local.load_state_dict(state)\n            local.train()\n            if method == "FedNova_Tail":\n                opt = torch.optim.SGD(local.parameters(), lr=cfg["lr"])\n                nsteps = max(1, min(4 * local_steps, round(local_steps * len(ids) / mean_client_len)))\n            elif method == "FedLogReg":\n                opt = torch.optim.Adam(local.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])\n                nsteps = local_steps\n            else:\n                opt = torch.optim.Adam(local.parameters(), lr=cfg["lr"])\n                nsteps = local_steps\n\n            for _ in range(nsteps):\n                b = r.choice(ids, min(B, len(ids)), replace=False)\n                opt.zero_grad()\n                logits = local(xx[b])\n                if method in ("FedAvg_Tail", "FedNova_Tail"):\n                    loss = tail_weighted_bce(logits, yy[b], lam=cfg["lam"], gamma=cfg["gamma"])\n                else:\n                    loss = nn.functional.binary_cross_entropy_with_logits(logits, yy[b])\n                loss.backward()\n                opt.step()\n                steps += 1\n\n            deltas.append({k: v.detach() - state[k] for k, v in local.state_dict().items()})\n            counts.append(nsteps)\n            comm += 2 * sum(v.numel() * v.element_size() for v in state.values())\n\n        if method == "FedNova_Tail":\n            tau_eff = float(np.dot(weights, counts))\n            avg = {k: tau_eff * sum(float(w / n) * a[k] for w, n, a in zip(weights, counts, deltas)) for k in state}\n        else:\n            avg = {k: sum(float(w) * a[k] for w, a in zip(weights, deltas)) for k in state}\n        model.load_state_dict({k: state[k] + avg[k] for k in state})\n\n        if rd % check == 0 or rd == rounds:\n            v = pauc(y[va], score_model(model, xx, va), 0.05)\n            if v > best + 1e-5:\n                best = v\n                best_round = rd\n                best_state = {k: val.detach().cpu().clone() for k, val in model.state_dict().items()}\n                wait = 0\n            else:\n                wait += 1\n        if rd >= min_round and wait >= patience:\n            break\n\n    model.load_state_dict(best_state)\n    pred = lambda ids: score_model(model, xx, ids)\n    info = dict(\n        seconds=time.perf_counter() - t0,\n        steps=steps,\n        bytes=comm,\n        best_round=best_round,\n        rounds_run=rd,\n        source_stop_pauc05=best,\n    )\n    return pred, None, info\n\n\ndef evaluate_bundle(folder: Path, held: str, s: int, method: str):\n    a = np.load(folder / "scores.npz", allow_pickle=False)\n    cy, cs, cd = a["cal_y"], a["cal_scores"], a["cal_d"]\n    sy, ss = a["source_test_y"], a["source_test_scores"]\n    ty_all, ts_all = a["target_y"], a["target_scores"]\n\n    # For FedBN_TTA, exclude the 2,000 unlabeled adaptation flows from evaluation (zero leakage)\n    if "tta_eval_mask" in a:\n        ev_mask = a["tta_eval_mask"].astype(bool)\n        ty, ts = ty_all[ev_mask], ts_all[ev_mask]\n    else:\n        ev_mask = np.ones(len(ty_all), dtype=bool)\n        ty, ts = ty_all, ts_all\n\n    alpha = 0.01\n    t_emp = emp_cut(cs[cy == 0], alpha)\n    t_fedrat = fedrat_cut(cs, cy, cd, alpha=alpha, delta=0.05)\n    t_oracle = emp_cut(ts[ty == 0], alpha)\n\n    locked_fpr, locked_rec, _, _ = eval_op(ty, ts, t_emp)\n    fedrat_fpr, fedrat_rec, _, _ = eval_op(ty, ts, t_fedrat)\n    _, oracle_rec, _, _ = eval_op(ty, ts, t_oracle)\n    src_fpr, _, _, _ = eval_op(sy, ss, t_emp)\n\n    # Zero-target-label unsupervised calibration at pi=1% (N_cal=2000, 5 disjoint draws)\n    neg_s = ts[ty == 0]\n    pos_s = ts[ty == 1]\n    rng = make_rng("day2_ut", held, s, method)\n    n_cal = min(2000, len(neg_s) // 2)\n    n_pos_cal = max(1, int(round(n_cal * 0.01)))\n    n_neg_cal = n_cal - n_pos_cal\n    ut_q_res, ut_rat_res = [], []\n    for _ in range(5):\n        cn = rng.choice(len(neg_s), size=n_neg_cal, replace=False)\n        cp = rng.choice(len(pos_s), size=min(n_pos_cal, len(pos_s) // 2), replace=False)\n        en_mask = np.ones(len(neg_s), dtype=bool)\n        en_mask[cn] = False\n        ep_mask = np.ones(len(pos_s), dtype=bool)\n        ep_mask[cp] = False\n        u_stream = np.concatenate([neg_s[cn], pos_s[cp]])\n        tq = emp_cut(u_stream, alpha)\n        tr = wilks_cut(u_stream, alpha, delta=0.05)\n        ev_n, ev_p = neg_s[en_mask], pos_s[ep_mask]\n        ut_q_res.append(((ev_n >= tq).mean(), (ev_p >= tq).mean(), float((ev_n >= tq).mean() <= alpha)))\n        ut_rat_res.append(((ev_n >= tr).mean(), (ev_p >= tr).mean(), float((ev_n >= tr).mean() <= alpha)))\n\n    ut_q_arr = np.array(ut_q_res)\n    ut_r_arr = np.array(ut_rat_res)\n\n    row = dict(\n        held=held,\n        seed=s,\n        method=method,\n        auroc=pauc(ty, ts),\n        pauc01=pauc(ty, ts, 0.01),\n        pauc05=pauc(ty, ts, 0.05),\n        ap=float(average_precision_score(ty, ts)),\n        source_test_fpr=src_fpr,\n        locked_fpr=locked_fpr,\n        locked_recall=locked_rec,\n        target_quantile_recall=oracle_rec,\n        fedrat_fpr=fedrat_fpr,\n        fedrat_recall=fedrat_rec,\n        ut_quantile_fpr=float(ut_q_arr[:, 0].mean()),\n        ut_quantile_recall=float(ut_q_arr[:, 1].mean()),\n        ut_quantile_cov=float(ut_q_arr[:, 2].mean()),\n        ut_rat_fpr=float(ut_r_arr[:, 0].mean()),\n        ut_rat_recall=float(ut_r_arr[:, 1].mean()),\n        ut_rat_cov=float(ut_r_arr[:, 2].mean()),\n    )\n\n    tta_pair = None\n    if "target_scores_pre_tta" in a:\n        ts_pre = a["target_scores_pre_tta"][ev_mask]\n        t_oracle_pre = emp_cut(ts_pre[ty == 0], alpha)\n        pre_fpr, pre_rec, _, _ = eval_op(ty, ts_pre, t_emp)\n        _, pre_oracle_rec, _, _ = eval_op(ty, ts_pre, t_oracle_pre)\n        tta_pair = dict(\n            held=held,\n            seed=s,\n            pre_auroc=pauc(ty, ts_pre),\n            post_auroc=row["auroc"],\n            pre_pauc01=pauc(ty, ts_pre, 0.01),\n            post_pauc01=row["pauc01"],\n            pre_locked_fpr=pre_fpr,\n            post_locked_fpr=locked_fpr,\n            pre_locked_recall=pre_rec,\n            post_locked_recall=locked_rec,\n            pre_oracle_recall=pre_oracle_rec,\n            post_oracle_recall=oracle_rec,\n            delta_oracle_recall=oracle_rec - pre_oracle_rec,\n            delta_locked_fpr=locked_fpr - pre_fpr,\n        )\n    return row, tta_pair\n\n\ndef run_all(input_path: Path, work: Path, methods=None, seeds=None, domains=None, smoke: bool = False):\n    work.mkdir(parents=True, exist_ok=True)\n    x, y, d, f = load_nf_cache(input_path, work)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    opts = dict(\n        device=device,\n        rounds=10 if smoke else 150,\n        min_round=5 if smoke else 30,\n        local_steps=2 if smoke else 8,\n        batch=256,\n        check_every=5,\n        patience=3 if smoke else 10,\n        trials=2 if smoke else 4,\n    )\n    methods = methods or METHODS\n    seeds = ([0] if smoke else SEEDS) if seeds is None else seeds\n    domains = (["NF-BoT-IoT"] if smoke else DOMAINS) if domains is None else domains\n    print(\n        f"Running FLIDS Day 2 on device={device} | methods={methods} | domains={len(domains)} | seeds={seeds}",\n        flush=True,\n    )\n\n    summary_rows = []\n    tta_rows = []\n\n    for held in domains:\n        for s in seeds:\n            fit, stop, cal, src_test, test = split_outer(y, d, [held], s)\n            for method in methods:\n                key = hashlib.sha256(repr(("day2_nf", held, s, method)).encode()).hexdigest()[:20]\n                folder = work / "bundles" / key\n                folder.mkdir(parents=True, exist_ok=True)\n                meta = folder / "done.json"\n\n                if not (meta.exists() and (folder / "scores.npz").exists()):\n                    candidates = candidate_grid(method)[: opts["trials"]]\n                    src_domains = sorted(set(d[fit]))\n                    inner = [\n                        (h, fit[d[fit] != h], stop[d[stop] != h], np.r_[fit[d[fit] == h], stop[d[stop] == h]])\n                        for h in src_domains\n                    ]\n                    trial_rows = []\n                    for j, cfg in enumerate(candidates):\n                        for h, a_idx, v_idx, b_idx in inner:\n                            tp = folder / f"trial_{j}_{hashlib.sha1(h.encode()).hexdigest()[:8]}.json"\n                            if tp.exists():\n                                tr_row = json.loads(tp.read_text())\n                            else:\n                                pred_fn, _, info = fit_method(x, y, d, a_idx, v_idx, method, cfg, make_seed(s, h, 0), opts)\n                                val_scores = pred_fn(b_idx)\n                                tr_row = dict(\n                                    candidate=j,\n                                    inner_holdout=h,\n                                    pauc05=pauc(y[b_idx], val_scores, 0.05),\n                                    pauc01=pauc(y[b_idx], val_scores, 0.01),\n                                    auroc=pauc(y[b_idx], val_scores),\n                                    config=cfg,\n                                    **info,\n                                )\n                                save_json(tp, tr_row)\n                            trial_rows.append(tr_row)\n\n                    chosen_id = int(\n                        pd.DataFrame(trial_rows)\n                        .groupby("candidate")["pauc05"]\n                        .mean()\n                        .sort_values(ascending=False, kind="stable")\n                        .index[0]\n                    )\n                    chosen = candidates[chosen_id]\n                    pd.DataFrame(trial_rows).to_csv(folder / "tuning.csv", index=False)\n\n                    pred_src, pred_tta, info = fit_method(x, y, d, fit, stop, method, chosen, make_seed(s, "final"), opts)\n                    scores = {}\n                    for role, ids in [("fit", fit), ("stop", stop), ("cal", cal), ("source_test", src_test)]:\n                        scores[f"{role}_ids"] = ids\n                        scores[f"{role}_y"] = y[ids]\n                        scores[f"{role}_d"] = d[ids].astype(str)\n                        scores[f"{role}_scores"] = pred_src(ids)\n\n                    scores["target_ids"] = test\n                    scores["target_y"] = y[test]\n                    scores["target_d"] = d[test].astype(str)\n                    scores["target_family"] = f[test].astype(str)\n\n                    if method == "FedBN_TTA":\n                        # Zero-leakage unsupervised AdaBN on a realistic 2,000-flow target stream at pi=1%\n                        scores["target_scores_pre_tta"] = pred_src(test)\n                        rng_tta = make_rng("day2_tta_stream", held, s)\n                        neg_t = test[y[test] == 0]\n                        pos_t = test[y[test] == 1]\n                        adapt_neg = rng_tta.choice(neg_t, size=min(1980, len(neg_t) // 2), replace=False)\n                        adapt_pos = rng_tta.choice(pos_t, size=min(20, len(pos_t) // 2), replace=False)\n                        adapt_ids = np.sort(np.concatenate([adapt_neg, adapt_pos]))\n                        eval_mask = ~np.isin(test, adapt_ids)\n                        scores["tta_adapt_ids"] = adapt_ids\n                        scores["tta_eval_mask"] = eval_mask\n                        scores["target_scores"] = pred_tta(test, adapt_ids=adapt_ids)\n                    else:\n                        scores["target_scores"] = pred_src(test)\n\n                    npwrite(folder / "scores.npz", **scores)\n                    save_json(\n                        meta,\n                        dict(\n                            held=held,\n                            seed=s,\n                            method=method,\n                            config=chosen,\n                            score_sha256=sha256_file(folder / "scores.npz"),\n                            **info,\n                        ),\n                    )\n                    print(f"DONE {held} | seed={s} | {method} | cfg={chosen} | {info[\'seconds\']:.1f}s", flush=True)\n\n                row, tta_pair = evaluate_bundle(folder, held, s, method)\n                summary_rows.append(row)\n                if tta_pair is not None:\n                    tta_rows.append(tta_pair)\n\n    df_op = pd.DataFrame(summary_rows)\n    df_op.to_csv(work / "day2_operating_runs.csv", index=False)\n\n    dom_means = df_op.groupby(["method", "held"]).mean(numeric_only=True).reset_index()\n    method_means = dom_means.groupby("method").mean(numeric_only=True).reset_index()\n    method_means.to_csv(work / "day2_method_means.csv", index=False)\n\n    if tta_rows:\n        pd.DataFrame(tta_rows).to_csv(work / "day2_fedbn_tta_paired.csv", index=False)\n\n    cols = [\n        "method", "auroc", "pauc01", "locked_fpr", "locked_recall",\n        "target_quantile_recall", "fedrat_fpr", "ut_quantile_fpr", "ut_quantile_recall", "ut_rat_cov",\n    ]\n    disp = method_means[cols].copy()\n    for c in cols[3:]:\n        disp[c] *= 100.0\n    print("\\n=== DAY 2 METHOD MEANS (Domain-Averaged, % for rates) ===")\n    print(disp.round(2).to_string(index=False))\n\n    out_zip = work / "FLIDS_Day2_Methods_Results.zip"\n    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for p in sorted(work.rglob("*")):\n            if p.is_file() and p != out_zip and p.name != "nf_cache.npz":\n                zf.write(p, arcname=str(p.relative_to(work)))\n    print(f"\\nExported archive: {out_zip} ({out_zip.stat().st_size / (1024 * 1024):.2f} MB)")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--input", required=True, type=Path, help="Path to FLIDS_Master_V3_Fresh_Results.zip, FLIDS_Exp09_Canonical_Input.zip, or nf_cache.npz")\n    ap.add_argument("--work", default=Path("./day2_work"), type=Path)\n    ap.add_argument("--smoke", action="store_true")\n    args = ap.parse_args()\n    run_all(args.input, args.work, smoke=args.smoke)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/kaggle/working/day2_methods.py').write_text(CODE, encoding='utf-8')
print('Wrote /kaggle/working/day2_methods.py', len(CODE), 'bytes')


In [ ]:
# Auto-discover input archive in /kaggle/input
from pathlib import Path
import torch

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def find_input():
    if INPUT_OVERRIDE:
        return Path(INPUT_OVERRIDE)
    candidates = [
        'FLIDS_Master_V3_Fresh_Results.zip',
        'FLIDS_Exp09_Canonical_Input.zip',
        'FLIDS_Experiment09_Results.zip',
        'nf_cache.npz',
    ]
    root = Path('/kaggle/input')
    for name in candidates:
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(
        'Could not find FLIDS_Master_V3_Fresh_Results.zip, FLIDS_Exp09_Canonical_Input.zip, or nf_cache.npz under /kaggle/input. '
        'Attach fragoa/notebook6ceac598dd or set INPUT_OVERRIDE.'
    )

INPUT_PATH = find_input()
print('Using input:', INPUT_PATH)


In [ ]:
# Run all Day 2 methods with checkpointing and export FLIDS_Day2_Methods_Results.zip
import sys
sys.path.insert(0, '/kaggle/working')
import day2_methods

day2_methods.run_all(
    input_path=INPUT_PATH,
    work=Path(WORK_DIR),
    methods=METHODS,
    smoke=SMOKE_TEST,
)


In [ ]:
# Display summary tables
import pandas as pd
from pathlib import Path

w = Path(WORK_DIR)
print('=== Method Means ===')
display(pd.read_csv(w / 'day2_method_means.csv'))
if (w / 'day2_fedbn_tta_paired.csv').exists():
    print('=== FedBN Pre-TTA vs Post-TTA Paired Comparison ===')
    df_tta = pd.read_csv(w / 'day2_fedbn_tta_paired.csv')
    display(df_tta.groupby('held')[['pre_locked_fpr','post_locked_fpr','pre_oracle_recall','post_oracle_recall','delta_oracle_recall']].mean().round(4))
